# CyberLab: Ataque, Defesa e Simulação de Malware

## 1️⃣ RECONHECIMENTO - Mapeamento de Superfície de Ataque

---

### 🎯 Objetivo

Realizar reconhecimento passivo e ativo do target para identificar:

- ✅ Portas abertas
- ✅ Serviços em execução (FTP, SSH, SMB, HTTP)
- ✅ Versões de software
- ✅ Potenciais vulnerabilidades
- ✅ Superfície de ataque disponível

### 📚 Conceitos

**Reconhecimento (Reconnaissance):**
- Primeira fase do ciclo de vida do ataque
- Coleta informações sobre o target
- Pode ser passivo (sem deixar rastros) ou ativo (deixa logs)

**Nmap:**
- Scanner de portas mais popular do mundo
- Identifica hosts, portas abertas, serviços
- Usa técnicas como SYN scan, UDP scan, NSE scripts

---

## 1. Setup Inicial

In [ ]:
import sys
import os
import json
from pathlib import Path
from datetime import datetime

# Adicionar scripts ao path
scripts_dir = Path("../scripts/python")
sys.path.insert(0, str(scripts_dir.absolute()))

from scanner import NmapScanner

# Carregamento de variáveis de ambiente
from dotenv import load_dotenv
load_dotenv(Path("../.env"))

# Configurações
TARGET_IP = os.getenv("TARGET_IP", "192.168.56.101")
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)

print(f"Target: {TARGET_IP}")
print(f"Resultados: {RESULTS_DIR}")
print(f"Timestamp: {datetime.now().isoformat()}")

## 2. Reconhecimento Passivo

Pesquisa sobre o target sem conectar diretamente a ele.

In [ ]:
# Reconhecimento passivo
# Em um ambiente real, usaríamos:
# - WHOIS lookup
# - DNS enumeration
# - Google dorking
# - Shodan search
# - Certificate transparency logs

passive_recon = {
    "target": TARGET_IP,
    "timestamp": datetime.now().isoformat(),
    "techniques": {
        "whois": "Informações de registro do IP",
        "dns": "Resolução de domínios",
        "ssl_cert": "Certificados SSL",
        "web_archive": "Histórico do site",
        "metadata": "Metadados de documentos"
    },
    "observations": [
        "Target é uma VM interna (192.168.56.0/24)",
        "IP privado, não acessível pela internet",
        "Propósito: Laboratório de cibersegurança"
    ]
}

print("Reconhecimento Passivo:")
for obs in passive_recon["observations"]:
    print(f"  • {obs}")

print("\n⚠️ Nota: Reconhecimento real seria mais completo em ambiente público")

## 3. Quick Scan (Top 1000 Portas)

In [ ]:
print(f"\n🔍 Iniciando Quick Scan em {TARGET_IP}...")
print("Escaneando as 1000 portas mais comuns...")
print("\nEste scan pode levar alguns minutos...\n")

scanner = NmapScanner(TARGET_IP, output_dir=RESULTS_DIR)

try:
    quick_results = scanner.quick_scan()
    print(json.dumps(quick_results, indent=2))
except Exception as e:
    print(f"❌ Erro: {e}")
    print("\n⚠️ Se Nmap não está instalado:")
    print("   Linux: sudo apt-get install nmap")
    print("   macOS: brew install nmap")
    print("   Windows: choco install nmap")

## 4. Análise de Resultados do Quick Scan

In [ ]:
# Análise interpretativa dos resultados
print("📊 Análise do Scan:\n")

# Serviços típicos descobertos em Metasploitable 2
expected_services = {
    21: {"service": "FTP", "risk": "ALTO", "description": "File Transfer Protocol - Vulnerável"},
    22: {"service": "SSH", "risk": "MÉDIO", "description": "Secure Shell - Possível força bruta"},
    23: {"service": "Telnet", "risk": "CRÍTICO", "description": "Telnet - Sem criptografia"},
    25: {"service": "SMTP", "risk": "MÉDIO", "description": "Simple Mail Transfer Protocol"},
    80: {"service": "HTTP", "risk": "MÉDIO", "description": "Web Server - DVWA"},
    111: {"service": "RPC", "risk": "ALTO", "description": "Remote Procedure Call"},
    139: {"service": "NetBIOS", "risk": "ALTO", "description": "Compartilhamento de arquivos"},
    445: {"service": "SMB", "risk": "CRÍTICO", "description": "Samba - Exploração possível"},
    3306: {"service": "MySQL", "risk": "ALTO", "description": "Database Server"},
    5432: {"service": "PostgreSQL", "risk": "ALTO", "description": "Database Server"}
}

print("Serviços Esperados em Metasploitable 2:\n")

for port, info in expected_services.items():
    print(f"Porta {port:5d} | {info['service']:12} | {info['risk']:7} | {info['description']}")

print("\n🎯 Serviços de Interesse para Ataque:")
print("  ✓ FTP (21) - Força bruta de credenciais")
print("  ✓ SSH (22) - Força bruta de credenciais")
print("  ✓ SMB (445) - Enumeração de shares e exploração")
print("  ✓ HTTP (80) - DVWA para testes de brute force web")
print("  ✓ MySQL (3306) - Possível sql injection")

## 5. Full Scan (Todas as Portas)

In [ ]:
print(f"\n🔍 Iniciando Full Scan em {TARGET_IP}...")
print("Escaneando todas as 65535 portas...")
print("\n⏱️ Este scan pode levar 10-30 minutos dependendo da rede...\n")

try:
    full_results = scanner.full_scan()
    print("✅ Full scan concluído!")
    print(json.dumps(full_results, indent=2)[:500] + "...")  # Primeiras 500 chars
except Exception as e:
    print(f"❌ Erro: {e}")

## 6. Vulnerability Scan (NSE Scripts)

In [ ]:
print(f"\n🔍 Iniciando Vulnerability Scan em {TARGET_IP}...")
print("Usando Nmap NSE Scripts para detectar vulnerabilidades...\n")

try:
    vuln_results = scanner.vuln_scan()
    print("✅ Vulnerability scan concluído!")
    print("\nExemplos de vulnerabilidades que podem ser detectadas:")
    print("  • SMB vulnerabilities (EternalBlue, etc)")
    print("  • SSL/TLS weak ciphers")
    print("  • Default credentials")
    print("  • Outdated software versions")
except Exception as e:
    print(f"❌ Erro: {e}")

## 7. Mapeamento de Enumeração

In [ ]:
# Planejar próximas etapas de enumeração
enumeration_plan = {
    "FTP (21)": {
        "tools": ["ftp", "nmap ftp-brute", "medusa"],
        "tests": [
            "Anonymous login",
            "Brute force credentials",
            "List directory contents"
        ]
    },
    "SSH (22)": {
        "tools": ["ssh-keyscan", "medusa", "hydra"],
        "tests": [
            "SSH version identification",
            "Brute force attack",
            "Weak algorithms detection"
        ]
    },
    "SMB (445)": {
        "tools": ["enum4linux", "smbclient", "nmap smb-*", "medusa"],
        "tests": [
            "Share enumeration",
            "User enumeration",
            "Brute force attack",
            "Null session testing"
        ]
    },
    "HTTP (80)": {
        "tools": ["curl", "burp suite", "nikto", "hydra"],
        "tests": [
            "Web application scanning",
            "DVWA brute force",
            "Directory enumeration",
            "SQL injection testing"
        ]
    }
}

print("\n📋 Plano de Enumeração por Serviço:\n")

for service, details in enumeration_plan.items():
    print(f"\n{service}")
    print(f"  Tools: {', '.join(details['tools'])}")
    print(f"  Tests:")
    for test in details['tests']:
        print(f"    • {test}")

## 8. Resumo e Próximos Passos

In [ ]:
print("\n" + "="*60)
print("✅ RECONHECIMENTO CONCLUÍDO!")
print("="*60)

print("\n📊 Sumário do Scan:")
print(f"  Target: {TARGET_IP}")
print(f"  Timestamp: {datetime.now().isoformat()}")
print(f"  Resultados salvos em: {RESULTS_DIR}")

print("\n📁 Arquivos Gerados:")
for file in RESULTS_DIR.glob("*.xml"):
    print(f"  • {file.name}")

print("\n🎯 Próximas Etapas:")
print("  1. Abra o notebook 2_preparacao_ataque.ipynb")
print("  2. Prepare wordlists e OSINT")
print("  3. Execute ataques de força bruta nos serviços identificados")

print("\n" + "="*60)
print("Lembre-se: Este é um ambiente EDUCACIONAL e ISOLADO!")
print("="*60)